In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/ex_8.py ──
"""
Shared infrastructure for Exercise 8 — Reinforcement Learning.

Contains: CartPole setup, reward plotting helpers, ExperimentTracker/ModelRegistry
setup, custom environment base class, evaluation utilities.
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
import random
from collections import deque
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

import gymnasium as gym
from gymnasium import spaces

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
device = get_device()

# Output directory for all visualisation artifacts
OUTPUT_DIR = Path("outputs") / "ex8_reinforcement_learning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# CARTPOLE ENVIRONMENT
# ════════════════════════════════════════════════════════════════════════


def make_cartpole() -> tuple[gym.Env, int, int]:
    """Create CartPole-v1 and return (env, obs_dim, n_actions)."""
    env = gym.make("CartPole-v1")
    obs_space = env.observation_space
    act_space = env.action_space
    assert (
        isinstance(obs_space, spaces.Box) and obs_space.shape is not None
    ), f"CartPole obs space expected gymnasium Box, got {type(obs_space).__name__}"
    assert isinstance(
        act_space, spaces.Discrete
    ), f"CartPole action space expected Discrete, got {type(act_space).__name__}"
    obs_dim = obs_space.shape[0]
    n_actions = int(act_space.n)
    print(f"CartPole-v1  obs_dim={obs_dim}  n_actions={n_actions}")
    return env, obs_dim, n_actions


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_rl.db"
    registry_db = "sqlite:///mlfp05_rl_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_reinforcement_learning", registry, True


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# REPLAY BUFFER — shared by DQN and custom env training
# ════════════════════════════════════════════════════════════════════════


class ReplayBuffer:
    """Fixed-size buffer storing (state, action, reward, next_state, done)."""

    def __init__(self, capacity: int = 10_000):
        self.buffer: deque = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        batch = random.sample(list(self.buffer), batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(np.array(states), dtype=torch.float32, device=device),
            torch.tensor(actions, dtype=torch.long, device=device),
            torch.tensor(rewards, dtype=torch.float32, device=device),
            torch.tensor(np.array(next_states), dtype=torch.float32, device=device),
            torch.tensor(dones, dtype=torch.float32, device=device),
        )

    def __len__(self):
        return len(self.buffer)


# ════════════════════════════════════════════════════════════════════════
# DQN NETWORK — shared by DQN training and custom env training
# ════════════════════════════════════════════════════════════════════════


class DQN(nn.Module):
    """Deep Q-Network: maps state -> Q-value for each action."""

    def __init__(self, obs_dim: int, n_actions: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ════════════════════════════════════════════════════════════════════════
# EVALUATION UTILITY
# ════════════════════════════════════════════════════════════════════════


def evaluate_policy(env: gym.Env, policy_fn, n_episodes: int = 30) -> list[float]:
    """Evaluate a policy function over n_episodes. Returns list of total rewards."""
    eval_returns: list[float] = []
    for i in range(n_episodes):
        state, _ = env.reset(seed=1000 + i)
        total = 0.0
        done = False
        while not done:
            action = policy_fn(state)
            state, reward, terminated, truncated, _ = env.step(action)
            total += float(reward)
            done = terminated or truncated
        eval_returns.append(total)
    return eval_returns


# ════════════════════════════════════════════════════════════════════════
# REWARD PLOTTING HELPERS
# ════════════════════════════════════════════════════════════════════════


def moving_average(xs: list[float], window: int = 10) -> list[float]:
    """Smooth a time series with a rolling mean."""
    if len(xs) < window:
        return xs
    arr = np.asarray(xs, dtype=np.float32)
    kernel = np.ones(window, dtype=np.float32) / window
    return list(np.convolve(arr, kernel, mode="valid"))


def plot_reward_curve(
    viz: ModelVisualizer,
    rewards: list[float],
    title: str,
    filename: str,
    window: int = 20,
    x_label: str = "Episode",
    y_label: str = "Reward",
) -> None:
    """Plot a reward curve with moving average and save to HTML."""
    metrics = {
        f"{title} reward": rewards,
        f"{title} moving avg ({window})": moving_average(rewards, window),
    }
    fig = viz.training_history(metrics=metrics, x_label=x_label, y_label=y_label)
    out_path = OUTPUT_DIR / filename
    fig.write_html(str(out_path))
    print(f"  Saved: {out_path}")


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION HELPER
# ════════════════════════════════════════════════════════════════════════


async def _register_rl_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    metrics_dict: dict[str, float],
):
    """Register a single RL policy network in ModelRegistry."""
    from kailash_ml.types import MetricSpec

    model_bytes = pickle.dumps(model.state_dict())
    metric_specs = [MetricSpec(name=k, value=v) for k, v in metrics_dict.items()]
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=metric_specs,
    )
    print(f"  Registered {name}: version={version.version}")
    return version


def register_rl_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    metrics_dict: dict[str, float],
):
    """Sync wrapper for RL model registration."""
    return asyncio.run(_register_rl_model(registry, name, model, metrics_dict))




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 8.2: Proximal Policy Optimization (PPO) from Scratch
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this exercise, you will be able to:
#   - Explain policy gradient intuition: "directly learning what to do"
#     rather than learning values and acting greedily
#   - Implement actor-critic architecture: "a coach and a scorekeeper"
#   - Implement Generalised Advantage Estimation (GAE): "crediting past
#     actions fairly" with low-variance advantage targets
#   - Implement the clipped surrogate objective: "don't change too much
#     at once" — the key to PPO's stability
#   - Train PPO on CartPole-v1 and track all metrics with ExperimentTracker
#   - Visualise reward curves, policy entropy, advantage distributions,
#     and actor vs critic loss
#   - Apply PPO to dynamic pricing for a Singapore ride-hailing platform
#
# PREREQUISITES: M5/ex_8/01_dqn.py (DQN concepts and CartPole).
# ESTIMATED TIME: ~40 min
# DATASETS: No static dataset — the environment IS the data source.
#   - CartPole-v1 (Gymnasium classic control, 4-D state, 2 actions)
#
# TASKS:
#   1. Understand PPO theory: policy gradients, actor-critic, GAE, clipping
#   2. Build PPO: actor-critic network, GAE computation, PPO update loop
#   3. Train PPO on CartPole-v1 with ExperimentTracker logging
#   4. Visualise agent behaviour: reward curve, policy entropy, advantage
#      distribution, actor vs critic loss
#   5. Apply: dynamic pricing for a Singapore ride-hailing platform
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

import gymnasium as gym
import polars as pl

from gymnasium import spaces
from kailash_ml import ModelVisualizer



## TASK 1 — Theory: Why PPO Exists


In [ ]:
# DQN learns Q-values (the value of each action) and then acts greedily.
# That's INDIRECT — first learn values, then derive a policy.
#
# PPO takes the DIRECT route: learn a policy pi(a|s) that maps states to
# action probabilities. No intermediate Q-values. This is called a
# POLICY GRADIENT method.
#
# Think of it like learning to drive:
#   DQN way: "First I'll learn the value of every possible steering angle
#            at every position on the road, then I'll always pick the
#            highest-value angle." (Indirect, requires memorising everything)
#   PPO way: "I'll learn a reflex: when I see a curve, turn the wheel
#            THIS much." (Direct, learns the action mapping)
#
# PPO (Schulman, 2017) has four key ideas:
#
#   (1) ACTOR-CRITIC — "A coach and a scorekeeper"
#       Actor: the policy network pi(a|s) — decides what to do
#       Critic: the value network V(s) — estimates how good a state is
#       They share a neural network trunk (efficient parameter sharing).
#       The critic's V(s) provides a BASELINE for variance reduction.
#
#   (2) GAE (Generalised Advantage Estimation) — "Crediting past actions"
#       Advantage A(s,a) = Q(s,a) - V(s) = "how much better was this
#       action than average?" GAE computes this with low variance:
#         delta_t = r_t + gamma * V(s_{t+1}) - V(s_t)
#         A_t = sum_{l=0}^{inf} (gamma * lambda)^l * delta_{t+l}
#       Lambda (0.95) trades off bias vs variance.
#
#   (3) CLIPPED SURROGATE OBJECTIVE — "Don't change too much at once"
#       The probability ratio r_t = pi_new(a|s) / pi_old(a|s) measures
#       how much the policy changed. PPO clips this ratio to [1-eps, 1+eps]
#       so the policy can't change drastically in one update.
#       L_clip = min(r_t * A_t, clip(r_t, 1-eps, 1+eps) * A_t)
#       This is the "proximal" in Proximal Policy Optimization.
#
#   (4) ENTROPY BONUS — "Keep exploring"
#       Add policy entropy to the loss: -0.01 * H(pi). This prevents
#       the policy from collapsing to always choosing one action.

print("=" * 70)
print("  TASK 1: PPO Theory — Policy Gradients with Stability Guarantees")
print("=" * 70)



## TASK 2 — Build PPO: actor-critic, GAE, clipped update


The shared trunk learns a common state representation. The actor head
    outputs action probabilities; the critic head outputs V(s).


Select action using current policy. Returns (action, log_prob, value).


Unlike DQN's replay buffer (random historical samples), PPO collects
    FRESH trajectories each iteration — on-policy learning.


GAE answers: "how much better was each action than what the critic
    expected?" Lambda controls the bias-variance tradeoff:
      lambda=0: only immediate TD error (high bias, low variance)
      lambda=1: full Monte Carlo return (low bias, high variance)
      lambda=0.95: sweet spot for most environments


Returns:
        (model, iter_returns, policy_entropies, actor_losses, critic_losses)


Sync wrapper for PPO training.


In [ ]:
print("\n" + "=" * 70)
print("  TASK 2: Build PPO from Scratch")
print("=" * 70)

# Set up CartPole and kailash engines
cartpole_env, obs_dim, n_actions = make_cartpole()
conn, tracker, exp_name, registry, has_registry = setup_engines()


class ActorCritic(nn.Module):

    def __init__(self, obs_dim: int, n_actions: int, hidden: int = 64):
        super().__init__()
        # TODO: Build the shared trunk — two hidden layers with Tanh activation
        # Hint: nn.Sequential(nn.Linear(obs_dim, hidden), nn.Tanh(),
        #   nn.Linear(hidden, hidden), nn.Tanh())
        self.trunk = # TODO
        # TODO: Create actor head (policy_head) and critic head (value_head)
        # Hint: policy_head = nn.Linear(hidden, n_actions)  — outputs action logits
        # Hint: value_head = nn.Linear(hidden, 1)  — outputs scalar state value
        self.policy_head = # TODO
        self.value_head = # TODO

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        h = self.trunk(x)
        return self.policy_head(h), self.value_head(h).squeeze(-1)

    def act(self, state: np.ndarray) -> tuple[int, torch.Tensor, torch.Tensor]:
        s = torch.from_numpy(state.astype(np.float32)).to(device)
        logits, value = self.forward(s)
        # TODO: Create a Categorical distribution from logits and sample an action
        # Hint: dist = Categorical(logits=logits); a = dist.sample()
        dist = # TODO
        a = # TODO
        return int(a.item()), dist.log_prob(a).detach(), value.detach()


def collect_trajectory(env: gym.Env, model: ActorCritic, max_steps: int):
    states, actions, log_probs, values, rewards, dones = [], [], [], [], [], []
    state, _ = env.reset(seed=int(np.random.randint(0, 100_000)))
    for _ in range(max_steps):
        action, log_prob, value = model.act(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        states.append(state.astype(np.float32))
        actions.append(action)
        log_probs.append(log_prob)
        values.append(value)
        rewards.append(float(reward))
        done = terminated or truncated
        dones.append(done)
        state = next_state
        if done:
            state, _ = env.reset(seed=int(np.random.randint(0, 100_000)))
    return states, actions, log_probs, values, rewards, dones


def compute_gae(
    rewards: list[float],
    values: list[torch.Tensor],
    dones: list[bool],
    gamma: float = 0.99,
    lam: float = 0.95,
) -> tuple[list[float], list[float]]:
    advantages = [0.0] * len(rewards)
    gae = 0.0
    next_value = 0.0
    # TODO: Compute GAE by iterating BACKWARDS through the trajectory
    # For each timestep t (in reverse):
    #   1. nonterminal = 1.0 - float(dones[t])
    #   2. delta = rewards[t] + gamma * next_value * nonterminal - float(values[t])
    #   3. gae = delta + gamma * lam * nonterminal * gae
    #   4. advantages[t] = gae
    #   5. next_value = float(values[t])
    for t in reversed(range(len(rewards))):
        nonterminal = # TODO
        delta = # TODO: TD error = reward + gamma * next_value * nonterminal - V(s)
        gae = # TODO: accumulate GAE = delta + gamma * lam * nonterminal * gae
        advantages[t] = gae
        next_value = float(values[t])
    returns = [a + float(v) for a, v in zip(advantages, values)]
    return advantages, returns


async def train_ppo_async(
    env: gym.Env,
    obs_d: int,
    n_act: int,
    n_iters: int = 30,
    steps_per_iter: int = 1024,
    epochs: int = 4,
    minibatch: int = 256,
    clip_eps: float = 0.2,
    lr: float = 3e-4,
    gamma: float = 0.99,
    lam: float = 0.95,
    run_name: str = "ppo_cartpole",
) -> tuple[ActorCritic, list[float], list[float], list[float], list[float]]:
    model = ActorCritic(obs_d, n_act).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    iter_returns: list[float] = []
    policy_entropies: list[float] = []
    actor_losses: list[float] = []
    critic_losses: list[float] = []

    async with tracker.track(experiment=exp_name, run_name=run_name) as run:
        await run.log_params(
            {
                "algorithm": "PPO",
                "gamma": str(gamma),
                "lambda": str(lam),
                "lr": str(lr),
                "clip_eps": str(clip_eps),
                "steps_per_iter": str(steps_per_iter),
                "epochs": str(epochs),
            }
        )

        print(f"\n== Training PPO ({run_name}) ==")
        for it in range(n_iters):
            # Collect fresh trajectory (on-policy)
            states, actions, old_log_probs, values, rewards, dones = collect_trajectory(
                env, model, steps_per_iter
            )
            advantages, returns = compute_gae(rewards, values, dones, gamma, lam)

            s_t = torch.tensor(np.stack(states), dtype=torch.float32, device=device)
            a_t = torch.tensor(actions, dtype=torch.long, device=device)
            old_lp_t = torch.stack(old_log_probs).to(device)
            adv_t = torch.tensor(advantages, dtype=torch.float32, device=device)
            ret_t = torch.tensor(returns, dtype=torch.float32, device=device)
            adv_t = (adv_t - adv_t.mean()) / (adv_t.std() + 1e-8)

            # PPO update: multiple epochs over the collected data
            n = s_t.size(0)
            idxs = np.arange(n)
            iter_actor_loss = 0.0
            iter_critic_loss = 0.0
            iter_entropy = 0.0
            update_count = 0

            for _ in range(epochs):
                np.random.shuffle(idxs)
                for start in range(0, n, minibatch):
                    mb = idxs[start : start + minibatch]
                    logits, vpred = model(s_t[mb])
                    dist = Categorical(logits=logits)
                    new_lp = dist.log_prob(a_t[mb])

                    # TODO: Compute the clipped surrogate objective
                    # 1. ratio = exp(new_log_prob - old_log_prob) — how much policy changed
                    # 2. surr1 = ratio * advantage — unclipped objective
                    # 3. surr2 = clamp(ratio, 1-clip_eps, 1+clip_eps) * advantage — clipped
                    # 4. policy_loss = -min(surr1, surr2).mean() — take the pessimistic bound
                    ratio = # TODO: torch.exp(new_lp - old_lp_t[mb])
                    surr1 = # TODO: ratio * adv_t[mb]
                    surr2 = # TODO: torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * adv_t[mb]
                    policy_loss = # TODO: -torch.min(surr1, surr2).mean()

                    # TODO: Compute value loss and entropy bonus
                    # Hint: value_loss = F.mse_loss(vpred, ret_t[mb])
                    # Hint: entropy = dist.entropy().mean()
                    value_loss = # TODO
                    entropy = # TODO

                    # TODO: Combine into total loss
                    # Hint: loss = policy_loss + 0.5 * value_loss - 0.01 * entropy
                    loss = # TODO

                    opt.zero_grad()
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                    opt.step()

                    iter_actor_loss += policy_loss.item()
                    iter_critic_loss += value_loss.item()
                    iter_entropy += entropy.item()
                    update_count += 1

            # Average episode return for this iteration
            ep_rs: list[float] = []
            running = 0.0
            for r, d in zip(rewards, dones):
                running += r
                if d:
                    ep_rs.append(running)
                    running = 0.0
            avg_ep_return = float(np.mean(ep_rs)) if ep_rs else float(running)
            iter_returns.append(avg_ep_return)

            avg_actor = iter_actor_loss / max(update_count, 1)
            avg_critic = iter_critic_loss / max(update_count, 1)
            avg_entropy = iter_entropy / max(update_count, 1)
            actor_losses.append(avg_actor)
            critic_losses.append(avg_critic)
            policy_entropies.append(avg_entropy)

            await run.log_metrics(
                {
                    "avg_episode_return": avg_ep_return,
                    "policy_entropy": avg_entropy,
                    "actor_loss": avg_actor,
                    "critic_loss": avg_critic,
                },
                step=it,
            )

            if (it + 1) % 10 == 0:
                print(
                    f"  iter {it+1:2d}  return={avg_ep_return:7.1f}  "
                    f"entropy={avg_entropy:.3f}  actor_loss={avg_actor:.4f}"
                )

        await run.log_metric("final_avg_return", iter_returns[-1])

    return model, iter_returns, policy_entropies, actor_losses, critic_losses


def train_ppo(
    env: gym.Env,
    obs_d: int,
    n_act: int,
    n_iters: int = 30,
    **kwargs,
) -> tuple[ActorCritic, list[float], list[float], list[float], list[float]]:
    return asyncio.run(train_ppo_async(env, obs_d, n_act, n_iters, **kwargs))



## TASK 3 — Train PPO on CartPole-v1


In [ ]:
print("\n" + "=" * 70)
print("  TASK 3: Train PPO on CartPole-v1")
print("=" * 70)

ppo_model, ppo_returns, ppo_entropies, ppo_actor_losses, ppo_critic_losses = train_ppo(
    cartpole_env, obs_dim, n_actions
)



### Checkpoint 1


In [ ]:
assert len(ppo_returns) == 30, "PPO should train for 30 iterations"
assert ppo_returns[-1] > 50.0, "PPO should achieve avg return > 50 by final iteration"
# INTERPRETATION: PPO learns a POLICY directly (probability of each action
# given a state), unlike DQN which learns Q-values and derives a policy.
# The clipped objective prevents the new policy from straying too far from
# the old one — this is the "proximal" in Proximal Policy Optimization.
# In M6, RLHF uses PPO to update an LLM's policy (word probabilities)
# using human preference as the reward signal.
print("--- Checkpoint 1 passed --- PPO trained on CartPole\n")



## TASK 4 — Visualise: reward curve, policy entropy, advantage distribution,
          actor vs critic loss


In [ ]:
print("=" * 70)
print("  TASK 4: Visualise PPO Agent Behaviour")
print("=" * 70)

viz = ModelVisualizer()



### Plot 1: PPO reward curve


In [ ]:
# TODO: Plot PPO returns using viz.training_history()
# Hint: metrics={"PPO avg episode return": ppo_returns}
fig1 = # TODO
fig1.write_html(str(OUTPUT_DIR / "02_ppo_reward_curve.html"))
print(f"  Saved: {OUTPUT_DIR / '02_ppo_reward_curve.html'}")



### Plot 2: Policy entropy over training


In [ ]:
# TODO: Plot entropy values using viz.training_history()
fig2 = # TODO
fig2.write_html(str(OUTPUT_DIR / "02_ppo_entropy.html"))
print(f"  Saved: {OUTPUT_DIR / '02_ppo_entropy.html'}")
# INTERPRETATION: Entropy measures how "spread out" the policy is across
# actions. High entropy = nearly uniform distribution (uncertain). Low
# entropy = confident in one action. Healthy training shows entropy
# DECREASING as the agent becomes more confident, but NOT collapsing to
# zero (which means it's stuck on one action regardless of state).



### Plot 3: Advantage distribution (first vs last iteration)


In [ ]:
# Re-collect a trajectory to show advantage distribution
states_final, _, _, values_final, rewards_final, dones_final = collect_trajectory(
    cartpole_env, ppo_model, 1024
)
advantages_final, _ = compute_gae(rewards_final, values_final, dones_final)

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# TODO: Create histogram of advantage values using go.Histogram
# Hint: go.Figure() with go.Histogram(x=advantages_final, nbinsx=50, ...)
fig3 = # TODO
fig3.update_layout(
    title="PPO Advantage Distribution (Trained Policy)",
    xaxis_title="Advantage A(s,a)",
    yaxis_title="Count",
)
fig3.write_html(str(OUTPUT_DIR / "02_ppo_advantage_dist.html"))
print(f"  Saved: {OUTPUT_DIR / '02_ppo_advantage_dist.html'}")
# INTERPRETATION: The advantage distribution shows how the critic evaluates
# actions. A distribution centred near zero with thin tails means the policy
# is well-calibrated — most actions are "about as good as expected." Fat
# positive tails mean some actions are surprisingly good (opportunities to
# improve the policy further).



### Plot 4: Actor loss vs critic loss


In [ ]:
# TODO: Plot actor and critic losses using viz.training_history()
# Hint: metrics={"Actor (policy) loss": ppo_actor_losses,
#   "Critic (value) loss": ppo_critic_losses}
fig4 = # TODO
fig4.write_html(str(OUTPUT_DIR / "02_ppo_actor_critic_loss.html"))
print(f"  Saved: {OUTPUT_DIR / '02_ppo_actor_critic_loss.html'}")
# INTERPRETATION: Two losses, two learning signals:
# - Actor loss: how much the policy improved this iteration
#   (should decrease then stabilise)
# - Critic loss: how accurate the value function is
#   (should decrease as V(s) predictions improve)
# If actor loss increases while return increases, the clipping is working
# — it's PREVENTING harmful updates.



### Checkpoint 2


In [ ]:
assert Path(OUTPUT_DIR / "02_ppo_reward_curve.html").exists()
assert Path(OUTPUT_DIR / "02_ppo_entropy.html").exists()
assert Path(OUTPUT_DIR / "02_ppo_advantage_dist.html").exists()
assert Path(OUTPUT_DIR / "02_ppo_actor_critic_loss.html").exists()
print("--- Checkpoint 2 passed --- all PPO visualisations generated\n")



## TASK 5 — Apply: Dynamic Pricing for a Singapore Ride-Hailing Platform


Models hourly pricing decisions across a 24-hour cycle.
    State (4,): [demand_level, supply_level, time_of_day, weather]
    Actions (5): price multiplier index (0.8x, 1.0x, 1.3x, 1.8x, 2.5x)
    Reward: revenue * customer_satisfaction
    Episode: 168 steps (hourly decisions for one week).


Singapore demand pattern: morning rush, lunch, evening rush, late night.


Driver availability: peaks during rush hours, dips late night.


In [ ]:
# SCENARIO: You're the pricing algorithms team at a Singapore ride-hailing
# platform (think Grab or Gojek). During peak hours (morning commute,
# evening rush, after-MRT-closure), demand spikes. You need to set a
# price multiplier that balances:
#   - Revenue: higher prices = more revenue per ride
#   - Customer satisfaction: too-high prices = riders switch to MRT/bus
#   - Driver supply: higher prices = more drivers come online
#
# State (4,): [demand_level, supply_level, time_of_day, weather]
#   - demand_level: current ride requests normalised [0, 1]
#   - supply_level: available drivers normalised [0, 1]
#   - time_of_day: cyclical encoding [0, 1] (0=midnight, 0.5=noon)
#   - weather: 0=clear, 0.5=rain, 1.0=heavy rain
#
# Actions (5): price multiplier levels
#   0=0.8x (discount), 1=1.0x, 2=1.3x, 3=1.8x, 4=2.5x (peak surge)
#
# Reward: revenue * satisfaction_factor
#   High prices boost revenue but reduce satisfaction. The agent must
#   find the sweet spot where revenue is high AND riders don't leave.

print("=" * 70)
print("  TASK 5: Apply PPO — Singapore Ride-Hailing Dynamic Pricing")
print("=" * 70)


class RideHailingPricingEnv(gym.Env):

    PRICE_MULTIPLIERS = [0.8, 1.0, 1.3, 1.8, 2.5]

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(4,), dtype=np.float32
        )
        self.action_space = spaces.Discrete(5)
        self.max_steps = 168  # one week of hourly decisions
        self.state = None
        self.step_count = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.step_count = 0
        time_of_day = self.np_random.uniform(0, 1)
        weather = 0.0  # clear
        demand = self._base_demand(time_of_day)
        supply = self._base_supply(time_of_day)
        self.state = np.array([demand, supply, time_of_day, weather], dtype=np.float32)
        return self.state.copy(), {}

    def _base_demand(self, time_norm: float) -> float:
        hour = time_norm * 24
        morning = 0.3 * np.exp(-0.5 * ((hour - 8) / 1.5) ** 2)
        lunch = 0.15 * np.exp(-0.5 * ((hour - 12.5) / 1.0) ** 2)
        evening = 0.35 * np.exp(-0.5 * ((hour - 18.5) / 1.5) ** 2)
        late = 0.2 * np.exp(-0.5 * ((hour - 0.5) / 1.5) ** 2)
        return np.clip(0.15 + morning + lunch + evening + late, 0.1, 1.0)

    def _base_supply(self, time_norm: float) -> float:
        hour = time_norm * 24
        if 6 <= hour < 10 or 17 <= hour < 21:
            return np.clip(0.6 + self.np_random.normal(0, 0.05), 0.3, 1.0)
        elif 1 <= hour < 6:
            return np.clip(0.2 + self.np_random.normal(0, 0.05), 0.1, 0.5)
        else:
            return np.clip(0.45 + self.np_random.normal(0, 0.05), 0.2, 0.8)

    def step(self, action):
        self.step_count += 1
        demand, supply, time_of_day, weather = self.state
        multiplier = self.PRICE_MULTIPLIERS[action]

        # TODO: Compute price elasticity effects on demand and supply
        # Hint: price_demand_effect = 1.0 - 0.3 * (multiplier - 1.0) — demand drops with price
        # Hint: price_supply_effect = 1.0 + 0.2 * (multiplier - 1.0) — supply rises with price
        # Then clip effective_demand and effective_supply to [0, 1]
        price_demand_effect = # TODO
        price_supply_effect = # TODO
        effective_demand = # TODO
        effective_supply = # TODO

        # TODO: Compute rides completed and revenue
        # Hint: rides_completed = min(effective_demand, effective_supply)
        # Hint: revenue = rides_completed * multiplier
        rides_completed = # TODO
        revenue = # TODO

        # TODO: Compute customer satisfaction based on price multiplier
        # Hint: satisfaction = 0.95 if multiplier <= 1.0, 0.85 if <= 1.3,
        #   0.65 - 0.1*weather if <= 1.8, else 0.4 - 0.15*weather
        if multiplier <= 1.0:
            satisfaction = 0.95
        elif multiplier <= 1.3:
            satisfaction = 0.85
        elif multiplier <= 1.8:
            satisfaction = # TODO
        else:
            satisfaction = # TODO

        # Reward: revenue weighted by satisfaction (long-term loyalty matters)
        reward = revenue * satisfaction

        # Advance time
        time_of_day = (time_of_day + 1.0 / 24.0) % 1.0

        # Weather changes (Singapore: sudden rain showers)
        if weather == 0 and self.np_random.random() < 0.15:
            weather = 0.5
        elif weather == 0.5 and self.np_random.random() < 0.2:
            weather = 1.0 if self.np_random.random() < 0.3 else 0.0
        elif weather == 1.0 and self.np_random.random() < 0.3:
            weather = 0.5

        # Next period demand and supply
        demand = np.clip(
            self._base_demand(time_of_day)
            + weather * 0.15
            + self.np_random.normal(0, 0.05),
            0.0,
            1.0,
        )
        supply = np.clip(
            self._base_supply(time_of_day)
            - weather * 0.1
            + self.np_random.normal(0, 0.05),
            0.0,
            1.0,
        )

        self.state = np.array([demand, supply, time_of_day, weather], dtype=np.float32)
        truncated = self.step_count >= self.max_steps
        return self.state.copy(), reward, False, truncated, {}


# Verify environment API
pricing_env = RideHailingPricingEnv()
obs, info = pricing_env.reset(seed=42)
assert obs.shape == (4,), "Pricing env should have 4-D state"
obs2, r, term, trunc, info = pricing_env.step(2)
assert isinstance(r, float), "Reward should be float"
print(f"  RideHailingPricing env: obs={obs.shape}, actions=5, sample_reward={r:.3f}")



### Train PPO on pricing environment


In [ ]:
ppo_pricing, pricing_returns, pricing_ent, pricing_al, pricing_cl = train_ppo(
    pricing_env,
    4,
    5,
    n_iters=40,
    steps_per_iter=1024,
    run_name="ppo_ride_hailing_pricing",
)



### Fixed-rule baseline


In [ ]:
def fixed_pricing_policy(state):
    demand = state[0]
    supply = state[1]
    gap = demand - supply
    if gap > 0.3:
        return 4  # 2.5x surge
    elif gap > 0.15:
        return 3  # 1.8x
    elif gap > 0.05:
        return 2  # 1.3x
    elif gap < -0.1:
        return 0  # 0.8x discount
    return 1  # 1.0x


baseline_pricing_returns = evaluate_policy(
    pricing_env, fixed_pricing_policy, n_episodes=30
)


def ppo_pricing_policy(state):
    with torch.no_grad():
        s = torch.from_numpy(state.astype(np.float32)).to(device)
        logits, _ = ppo_pricing(s)
        return int(logits.argmax().item())


ppo_pricing_returns = evaluate_policy(pricing_env, ppo_pricing_policy, n_episodes=30)

print(
    f"\n  Fixed-rule baseline:  mean weekly revenue*satisfaction = {np.mean(baseline_pricing_returns):.1f}"
)
print(
    f"  PPO learned policy:   mean weekly revenue*satisfaction = {np.mean(ppo_pricing_returns):.1f}"
)

revenue_improvement = float(np.mean(ppo_pricing_returns)) - float(
    np.mean(baseline_pricing_returns)
)
revenue_pct = (
    revenue_improvement / abs(float(np.mean(baseline_pricing_returns))) * 100
    if float(np.mean(baseline_pricing_returns)) != 0
    else 0
)
print(f"  Improvement: {revenue_improvement:+.1f} ({revenue_pct:+.1f}%)")



### Visualise: PPO vs fixed-rule pricing


In [ ]:
# TODO: Create a box plot comparing PPO vs fixed-rule weekly performance
# Hint: pl.DataFrame with "Policy" and "Weekly Revenue x Satisfaction" columns
pricing_comparison_df = # TODO
fig_apply = # TODO: viz.box_plot(...)
fig_apply.write_html(str(OUTPUT_DIR / "02_ppo_pricing_comparison.html"))
print(f"  Saved: {OUTPUT_DIR / '02_ppo_pricing_comparison.html'}")



### Visualise: learned pricing across demand levels


In [ ]:
demand_levels = np.linspace(0.1, 1.0, 20)
multiplier_names = ["0.8x", "1.0x", "1.3x", "1.8x", "2.5x"]
pricing_decisions = []
ppo_pricing.eval()
for dl in demand_levels:
    # Evening rush hour (time=0.77), moderate supply (0.5), clear weather
    state = torch.tensor([dl, 0.5, 0.77, 0.0], dtype=torch.float32, device=device)
    with torch.no_grad():
        logits, _ = ppo_pricing(state)
        action = int(logits.argmax().item())
    pricing_decisions.append(multiplier_names[action])

print("\n  Learned Pricing (evening rush, supply=0.5, clear weather):")
for dl, dec in zip(demand_levels[::4], pricing_decisions[::4]):
    print(f"    Demand={dl:.2f} -> {dec}")

# INTERPRETATION: PPO learns a pricing policy that adapts to demand-supply
# dynamics AND customer sensitivity. Unlike fixed rules that only look at
# the demand-supply gap, PPO considers time of day (commuters tolerate
# moderate surges, late-night riders are more price-sensitive) and weather
# (rain increases demand but also increases price sensitivity). The net
# result: higher revenue with less customer churn.

pricing_env.close()



### Checkpoint 3


In [ ]:
assert len(pricing_returns) == 40, "Pricing PPO should train for 40 iterations"
assert Path(OUTPUT_DIR / "02_ppo_pricing_comparison.html").exists()
print("--- Checkpoint 3 passed --- PPO applied to ride-hailing pricing\n")



### Register PPO models


In [ ]:
if has_registry:
    register_rl_model(
        registry,
        "m5_ppo_cartpole",
        ppo_model,
        {
            "avg_return_final": float(ppo_returns[-1]),
            "algorithm": 1.0,
            "iterations_trained": float(len(ppo_returns)),
        },
    )
    register_rl_model(
        registry,
        "m5_ppo_ride_hailing",
        ppo_pricing,
        {
            "avg_weekly_reward": float(np.mean(pricing_returns[-10:])),
            "iterations_trained": 40.0,
        },
    )

cartpole_env.close()

# Clean up
await conn.close()



## REFLECTION


[x] Built PPO from scratch: actor-critic architecture with shared trunk
  [x] Implemented GAE for low-variance advantage estimation
  [x] Implemented the clipped surrogate objective for stable updates
  [x] Trained PPO on CartPole-v1 — steadier learning than DQN
  [x] Visualised the agent's learning:
      - Reward curve: smoother convergence than DQN
      - Policy entropy: confidence increasing over training
      - Advantage distribution: well-calibrated action evaluation
      - Actor vs critic loss: two learning signals working in tandem
  [x] Applied PPO to Singapore ride-hailing dynamic pricing:
      - Built a custom environment with realistic demand patterns
        (morning rush, evening peak, late-night after MRT closure)
      - PPO learned to balance revenue and customer satisfaction
      - Visualised pricing decisions across demand levels

  KEY INSIGHT:
  PPO learns WHAT TO DO directly (policy), while DQN learns HOW GOOD
  each action IS (values). PPO is more stable and handles complex
  action spaces better. This is exactly why RLHF (in M6) uses PPO —
  it can fine-tune an LLM's "policy" (next-token probabilities) based
  on human preference rewards.

  BRIDGE TO M6 (RLHF):
  In RLHF, the "environment" is text generation, the "state" is the
  prompt + tokens so far, the "action" is the next token, and the
  "reward" comes from a preference model trained on human rankings.
  DPO (Direct Preference Optimization) achieves the same goal without
  needing a separate reward model.

  Next: In 03_custom_environments.py, you'll build 5 business-themed
  environments that model real Singapore decision problems.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — PPO")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — five instruments before Visualise


In [ ]:
# Reference: `kailash_ml.diagnostics` (via `kailash-ml`) — see gold standard
# `solutions/ex_1/01_standard_ae.py` for the full pattern.
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _diag_loss(m, batch):
    # PPO clipped objective + value loss
    # Customise per your exercise's loss shape.
    if isinstance(batch, (tuple, list)):
        x = batch[0]
        y = batch[1] if len(batch) > 1 else None
    else:
        x, y = batch, None
    out = m(x)
    import torch.nn.functional as F
    if y is None:
        return F.mse_loss(out, x)
    return F.cross_entropy(out, y)


print("\n── Diagnostic Report (PPO — Proximal Policy Optimization) ──")
try:
    diag, findings = run_diagnostic_checkpoint(
        actor_critic,
        rollout_loader,
        _diag_loss,
        title="PPO — Proximal Policy Optimization",
        n_batches=8,
        show=False,
    )
except Exception as exc:
    # Diagnostic is pedagogical — never block the exercise on it.
    print(f"[diagnostic skipped: {exc}]")

# ══════ EXPECTED OUTPUT (synthesized reference — full run produces similar pattern) ══════



## DL Diagnostics Report — Prescription Pad


In [ ]:
# [✓] Gradient flow (HEALTHY): RMS 2.1e-03 across actor and critic heads.
#     PPO clipping keeps update ratio in [0.8, 1.2] — stable by design.
# [!] Policy entropy collapsing at epoch 8 — early sign of premature convergence.
# [✓] Reward curve: steady climb, no collapse events.



## STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:


In [ ]:
#  [BLOOD TEST — PPO-SPECIFIC] The clipped objective is what keeps
#     PPO stable. update_ratio > 1.2 or < 0.8 would mean the policy
#     is moving too fast — but the clip prevents it. That's WHY
#     PPO dominates in 2024+ (slide 5.8).
#
#  [X-RAY — POLICY ENTROPY] Entropy collapse means the policy is
#     becoming deterministic — no exploration, no learning new
#     strategies. Slide 5.8 Prescription Pad: add entropy bonus
#     (coef ~0.01), or use SAC which has entropy regularisation
#     baked in.
#     >> Prescription: raise entropy coefficient from 0.01 → 0.05
#        to encourage exploration.

